# Dataset building

**Goal:** turn 11 nights of images into a clean, split, normalized, balanced dataset, and understand every
processing step between raw pixels and model input.

Read `docs/dataset_building.md` alongside this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.query_ztf_metadata import query_ztf_metadata
from asthunt.dataset.build_night_detections import build_night_detections
from asthunt.dataset.normalize_stamps import normalize_stamps
from asthunt.dataset.augment_stamps import augment_stamps
from asthunt.dataset.balance_classes import balance_classes
from asthunt.dataset.assemble_stamp_dataset import assemble_stamp_dataset
from asthunt.evaluate.check_split_leakage import check_split_leakage

project = load_config("project"); paths = project["paths"]
acquire, inject, detect, dataset = (load_config(name) for name in ["acquire", "inject", "detect", "dataset"])


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


metadata = load_unit_metadata(unit_of(None), paths["raw"])

## 1. Build every night (cached: re-running is instant)
Each night: load → inject 300 fakes → detect → label → stamps → save 3 files. The random generator is seeded with the
project seed **and the night's date**, so every night gets different fakes but re-runs are identical.

In [ ]:
summary = []
for split, nights in acquire["nights"].items():
    for night in nights:
        rng = np.random.default_rng([project["seed"], int(night.replace("-", ""))])
        detections, stamps, injections = build_night_detections(unit_of(night), metadata, acquire, inject, detect, paths, rng)
        summary.append({"split": split, "night": night, "detections": len(detections),
                        "real": int(detections["label"].sum()),
                        "known_asteroid_hits": int((detections["known_asteroid"] != "").sum()),
                        "seeing_px": detections["seeing"].median(), "maglim": detections["maglim"].median()})
summary = pd.DataFrame(summary)
summary["bogus_per_real"] = ((summary["detections"] - summary["real"]) / summary["real"]).round(0)
summary

## 2. Leakage check: no night in two splits

In [ ]:
nights = acquire["nights"]
print("train vs validation:", check_split_leakage(nights["train"], nights["validation"]))
print("train vs test      :", check_split_leakage(nights["train"], nights["test"]))
print("validation vs test :", check_split_leakage(nights["validation"], nights["test"]))

## 3. Explore the training data (EDA)
Before any model: how do real and bogus detections differ in the simple measured numbers?

In [ ]:
train_tables = [pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(night))}.parquet") for night in nights["train"]]
train_rows = pd.concat(train_tables, ignore_index=True)

features = ["snr", "npix", "a", "b", "peak"]
fig, axes = plt.subplots(1, len(features), figsize=(20, 3.5))
for ax, feature in zip(axes, features):
    values = train_rows[feature].clip(*np.percentile(train_rows[feature], [0.5, 99.5]))
    bins = np.linspace(values.min(), values.max(), 60)
    for label, name in [(0, "bogus"), (1, "real")]:
        ax.hist(values[train_rows["label"] == label], bins=bins, alpha=0.6, density=True, label=name)
    ax.set_title(feature)
axes[0].legend(); plt.show()

Real detections are brighter (higher SNR, peak) and bigger (more pixels) on average, but the distributions **overlap**
heavily at the faint end, where it matters. Simple features won't be enough: the model will need to look at the pixels.

## 4. Normalization: same scale for every night
Raw stamps carry each night's sky level and noise. Normalizing turns pixels into "sigmas above the sky".

In [ ]:
first_night, second_night = nights["train"][0], nights["train"][-1]
raw_a = np.load(paths["processed"] / f"stamps__{make_unit_tag(unit_of(first_night))}.npy", mmap_mode="r")[:2000].astype(np.float32)
raw_b = np.load(paths["processed"] / f"stamps__{make_unit_tag(unit_of(second_night))}.npy", mmap_mode="r")[:2000].astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for raw, night in [(raw_a, first_night), (raw_b, second_night)]:
    axes[0].hist(raw[:, 0].ravel(), bins=200, range=(0, 2000), histtype="step", density=True, label=night)
    axes[1].hist(normalize_stamps(raw)[:, 0].ravel(), bins=200, range=(-6, 10), histtype="step", density=True, label=night)
axes[0].set_title("science channel, raw counts"); axes[1].set_title("science channel, normalized (sigmas)")
axes[0].legend(); plt.show()

Raw: two different nights sit at different sky levels. Normalized: both are centred on 0 with width 1.

## 5. Augmentation: one stamp, eight orientations

In [ ]:
detections = pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(first_night))}.parquet")
one_real = normalize_stamps(raw_a[[int(np.flatnonzero(detections["label"].to_numpy()[:2000] == 1)[0])]])
rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax in axes:
    ax.imshow(augment_stamps(one_real, rng)[0, 2], origin="lower", cmap="gray"); ax.axis("off")
plt.suptitle("difference channel of one real detection, randomly rotated / mirrored"); plt.show()

## 6. Balance and assemble
- **Training:** all real + 5 bogus per real (the model must notice the rare class).
- **Validation:** *everything*, with the true imbalance, so its scores tell us how the model behaves on a real night.

In [ ]:
rng = np.random.default_rng(project["seed"])
train_stamps, train_labels, train_table = assemble_stamp_dataset([unit_of(n) for n in nights["train"]], paths["processed"], rng,
                                                                  bogus_per_real=dataset["bogus_per_real"])
validation_stamps, validation_labels, validation_table = assemble_stamp_dataset([unit_of(n) for n in nights["validation"]], paths["processed"], rng)

for name, labels in [("train", train_labels), ("validation", validation_labels)]:
    print(f"{name:10s}: {len(labels):7d} stamps, {labels.sum():6d} real ({labels.mean():.1%})")
print("stamp array shape:", train_stamps.shape, train_stamps.dtype, f"{train_stamps.nbytes / 1e9:.2f} GB")

## 7. What's on disk (data versioning by name)

In [ ]:
for f in sorted(paths["processed"].glob("*")):
    print(f"{f.stat().st_size / 1e6:8.1f} MB  {f.name}")

## Exercises
1. Which training night has the most bogus per real detection? Look at its seeing and maglimit: why might that be?
2. Change `bogus_per_real` to 1 and to 20. How do the class fractions change? Predict what that will do to a model's scores.
3. Why must augmentation never be applied to validation data?